# Part 14 — LangGraph: State, Graphs & Agentic Workflows

LangGraph models agent workflows as **directed graphs** where state flows through nodes connected by edges. Supports cycles, branching, persistence, and human-in-the-loop — things a simple chain cannot express.

```
GRAPH EXECUTION MODEL
══════════════════════════════════════════════════════════════════════

COMPILE TIME (once)                    RUNTIME (per invocation)
─────────────────────                  ─────────────────────────────
StateGraph(State)                      graph.invoke({"messages": [...]})
  .add_node("a", fn_a)                        │
  .add_node("b", fn_b)                        ▼
  .add_edge(START, "a")              ┌─────────────────┐
  .add_conditional_edges(            │  Initial State   │
       "a", router, {                └────────┬────────┘
         "x": "b",                            │ START edge
         "y": END                             ▼
       })                            ┌─────────────────┐
  .compile()                         │   Node "a"      │  fn_a(state) → dict
         │                           │  receives state │
         ▼                           │  returns update │
  compiled graph                     └────────┬────────┘
  (immutable DAG/cyclic)                      │ conditional edge
                                      ┌───────┴────────┐
                                      ▼                 ▼
                               ┌──────────┐        ┌───────┐
                               │ Node "b" │        │  END  │
                               └──────────┘        └───────┘
```

| Section | Content |
|---|---|
| 1 | Core primitives — State, Nodes, Edges, reducers |
| 2 | Defining State — TypedDict, Pydantic, custom reducers |
| 3 | Building a graph — 5-step process |
| 4 | Execution model visualization |
| 5 | Tool integration — `@tool`, `ToolNode`, `tools_condition` |
| 6 | Conditional routing — `add_conditional_edges` |
| 7 | Graph topology visualization |
| 8 | Memory & checkpointing — `MemorySaver`, `SqliteSaver`, `thread_id` |
| 9 | Time-travel debugging — `get_state_history`, checkpoint replay |
| 10 | Human-in-the-loop — `interrupt()` |
| 11 | Summary |

## 1 — Core Primitives

| Primitive | Type | Responsibility | Returns |
|---|---|---|---|
| **State** | `TypedDict` or `BaseModel` | Shared data structure flowing through the graph | — |
| **Reducer** | `Annotated[T, fn]` | Merges a node's partial update into the existing state field | merged value |
| **Node** | `Callable[[State], dict]` | Executes agent logic (LLM call, tool call, transform) | partial state update |
| **Fixed edge** | `add_edge(src, dst)` | Unconditional transition — always goes to `dst` after `src` | — |
| **Conditional edge** | `add_conditional_edges(src, fn, mapping)` | `fn(state)` returns a key; `mapping[key]` selects next node | key → node name |
| **START** | sentinel | Entry point — first edge must leave from `START` | — |
| **END** | sentinel | Terminal node — graph halts when reached | — |
| **Compiled graph** | `CompiledGraph` | Immutable executable produced by `.compile()` | — |

**State update semantics:**  
A node returns a `dict` with only the fields it changed. LangGraph applies each field's reducer to merge the update:
- Default reducer: **replace** — `new_val` overwrites `old_val`
- `add_messages` reducer: **append** — new messages are appended to the list, never replaced
- Custom reducer: any `fn(old, new) → merged` function

**Two execution phases:**
1. **Compile time** — `StateGraph` + nodes + edges → `CompiledGraph` (validates topology, detects cycles)
2. **Runtime** — `graph.invoke(state)` executes the graph, propagating state updates step by step

In [ ]:
from typing import Annotated, TypedDict, Literal
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.graph.message import add_messages
from pydantic import BaseModel

# ── Option 1: TypedDict (lightweight, no validation) ─────────────────────────
class SimpleState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    # add_messages: appends new messages instead of replacing the list

# ── Option 2: Pydantic BaseModel (type validation + defaults) ────────────────
class AgentState(BaseModel):
    messages: Annotated[list[BaseMessage], add_messages] = []
    current_task: str = ""
    iteration_count: int = 0
    is_complete: bool = False

# ── Custom reducers ───────────────────────────────────────────────────────────
def keep_latest(existing: str, new: str) -> str:
    """Replace — default behavior for non-list fields."""
    return new

def accumulate(existing: list, new: list) -> list:
    """Append — useful for collecting results across nodes."""
    return existing + new

def take_max(existing: int, new: int) -> int:
    """Custom merge — e.g., for tracking max iteration count."""
    return max(existing, new)

class ResearchState(TypedDict):
    messages:       Annotated[list[BaseMessage], add_messages]  # append
    search_results: Annotated[list[str], accumulate]            # accumulate across iterations
    best_score:     Annotated[int, take_max]                    # keep highest
    final_report:   str                                          # replace (default)

# ── Demonstrate reducer behavior ─────────────────────────────────────────────
from langgraph.graph.message import add_messages

existing_msgs = [HumanMessage(content="Hello")]
new_msgs      = [AIMessage(content="Hi there")]
merged        = add_messages(existing_msgs, new_msgs)

print("add_messages reducer:")
print(f"  existing : {[m.content for m in existing_msgs]}")
print(f"  new      : {[m.content for m in new_msgs]}")
print(f"  merged   : {[m.content for m in merged]}")
print()
print("Default replace reducer:")
print(f"  keep_latest('old_task', 'new_task') → '{keep_latest('old_task', 'new_task')}'")
print()
print("Custom accumulate reducer:")
print(f"  accumulate(['result-1'], ['result-2', 'result-3']) → {accumulate(['result-1'], ['result-2', 'result-3'])}")

## 3 — Building a Graph: 5-Step Process

```
Step 1: Define State class        →  TypedDict or BaseModel
Step 2: Create StateGraph(State)  →  graph builder
Step 3: Add nodes                 →  builder.add_node("name", fn)
Step 4: Add edges                 →  builder.add_edge / add_conditional_edges
Step 5: Compile                   →  graph = builder.compile()
                                      then: graph.invoke(initial_state)
```

A node function signature is always `fn(state: State) -> dict`. The returned `dict` contains only the fields being updated — not the full state.

In [ ]:
import os
from typing import Annotated, TypedDict
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.environ.get("OPENAI_API_KEY"))

# ── Step 1: Define State ──────────────────────────────────────────────────────
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# ── Step 2: Graph builder ─────────────────────────────────────────────────────
builder = StateGraph(ChatState)

# ── Step 3: Add nodes ─────────────────────────────────────────────────────────
def chat_node(state: ChatState) -> dict:
    """Single LLM call node. Receives full state, returns only updated fields."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}   # add_messages reducer appends this

builder.add_node("chat", chat_node)

# ── Step 4: Add edges ─────────────────────────────────────────────────────────
builder.add_edge(START, "chat")   # START → chat (always)
builder.add_edge("chat",  END)    # chat  → END  (always)

# ── Step 5: Compile ───────────────────────────────────────────────────────────
graph = builder.compile()

# ── Invoke ────────────────────────────────────────────────────────────────────
result = graph.invoke({
    "messages": [HumanMessage(content="What are the three main components of LangGraph?")]
})

print("Messages in final state:")
for msg in result["messages"]:
    role = "Human" if isinstance(msg, HumanMessage) else "AI"
    print(f"  [{role}] {msg.content[:120]}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

os.makedirs("images", exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.patch.set_facecolor('#0f0f0f')
for ax in axes:
    ax.set_facecolor('#1a1a2e')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')

def draw_box(ax, x, y, w, h, label, fc, ec, tc='white', fontsize=9, bold=False):
    box = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
        boxstyle="round,pad=0.12", linewidth=1.5, edgecolor=ec, facecolor=fc)
    ax.add_patch(box)
    ax.text(x, y, label, color=tc, fontsize=fontsize, ha='center', va='center',
        fontweight='bold' if bold else 'normal')

def arrow(ax, x0, y0, x1, y1, color='#666', lw=1.5, label='', label_offset=(0.1, 0)):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
        arrowprops=dict(arrowstyle='->', color=color, lw=lw))
    if label:
        mx, my = (x0 + x1)/2 + label_offset[0], (y0 + y1)/2 + label_offset[1]
        ax.text(mx, my, label, color='#888', fontsize=7.5, ha='left', va='center')

# ── Panel 1: Linear Chain ────────────────────────────────────────────────────
ax = axes[0]
ax.set_title("Linear Chain", color='white', fontsize=12, fontweight='bold', pad=8)

nodes_linear = [("START", 5, 9.0, '#1e3a5f', '#4a9eff'),
                ("load_docs", 5, 7.2, '#2d4a1e', '#90c47a'),
                ("chunk", 5, 5.4, '#2d4a1e', '#90c47a'),
                ("embed", 5, 3.6, '#2d4a1e', '#90c47a'),
                ("END", 5, 1.8, '#3a1e1e', '#ff6060')]
for lbl, x, y, fc, ec in nodes_linear:
    draw_box(ax, x, y, 2.8, 0.7, lbl, fc, ec, bold=True)

for i in range(len(nodes_linear) - 1):
    arrow(ax, nodes_linear[i][1], nodes_linear[i][2] - 0.35,
             nodes_linear[i+1][1], nodes_linear[i+1][2] + 0.35)

ax.text(5, 0.5, "No branching, no cycles\nUse: ETL pipelines",
    color='#888', fontsize=8, ha='center', va='center')

# ── Panel 2: Branching Router ────────────────────────────────────────────────
ax = axes[1]
ax.set_title("Branching Router", color='white', fontsize=12, fontweight='bold', pad=8)

draw_box(ax, 5, 9.0, 2.8, 0.7, "START", '#1e3a5f', '#4a9eff', bold=True)
draw_box(ax, 5, 7.2, 2.8, 0.7, "classify", '#3a2a10', '#f0a030', bold=True)
draw_box(ax, 2.0, 5.0, 2.2, 0.7, "code_expert", '#2d4a1e', '#90c47a', fontsize=8)
draw_box(ax, 5.0, 5.0, 2.2, 0.7, "research", '#2d4a1e', '#90c47a', fontsize=8)
draw_box(ax, 8.0, 5.0, 2.2, 0.7, "general", '#2d4a1e', '#90c47a', fontsize=8)
draw_box(ax, 5, 2.8, 2.8, 0.7, "END", '#3a1e1e', '#ff6060', bold=True)

arrow(ax, 5, 8.65, 5, 7.55)
# classify → branches
ax.plot([5, 2.0], [6.85, 5.35], color='#f0a030', lw=1.5)
ax.annotate('', xy=(2.0, 5.35), xytext=(2.0, 5.36),
    arrowprops=dict(arrowstyle='->', color='#f0a030', lw=1.5))
ax.plot([5, 5.0], [6.85, 5.35], color='#f0a030', lw=1.5)
ax.plot([5, 8.0], [6.85, 5.35], color='#f0a030', lw=1.5)

for x in [2.0, 5.0, 8.0]:
    arrow(ax, x, 4.65, 5, 3.15, color='#666')

ax.text(6.3, 6.2, "conditional\nedge", color='#f0a030', fontsize=7.5)
ax.text(5, 1.8, "Use: task routing, multi-agent dispatch",
    color='#888', fontsize=8, ha='center', va='center')

# ── Panel 3: ReAct Loop (Agent + Tools) ──────────────────────────────────────
ax = axes[2]
ax.set_title("ReAct Loop (Agent + Tools)", color='white', fontsize=12, fontweight='bold', pad=8)

draw_box(ax, 5, 9.2, 2.8, 0.7, "START", '#1e3a5f', '#4a9eff', bold=True)
draw_box(ax, 5, 7.4, 3.0, 0.7, "agent\n(LLM + tools_condition)", '#2a1a4a', '#a060ff', fontsize=8)
draw_box(ax, 5, 5.2, 2.8, 0.7, "tool_node\n(executes tools)", '#3a2a10', '#f0a030', fontsize=8)
draw_box(ax, 5, 3.0, 2.8, 0.7, "END", '#3a1e1e', '#ff6060', bold=True)

arrow(ax, 5, 8.85, 5, 7.75)
# agent → tool_node
ax.annotate('', xy=(5, 5.55), xytext=(5, 7.05),
    arrowprops=dict(arrowstyle='->', color='#f0a030', lw=1.5))
ax.text(5.15, 6.3, "tool_call\nin response", color='#f0a030', fontsize=7.5)

# tool_node → agent (loop back)
ax.annotate('', xy=(3.2, 7.4), xytext=(3.2, 5.2),
    arrowprops=dict(arrowstyle='->', color='#60c0c0', lw=1.5,
        connectionstyle="arc3,rad=0.0"))
ax.plot([3.5, 3.2], [5.2, 5.2], color='#60c0c0', lw=1.5)
ax.plot([3.5, 3.2], [7.4, 7.4], color='#60c0c0', lw=1.5)
ax.text(1.5, 6.3, "loop\nback", color='#60c0c0', fontsize=7.5, ha='center')

# agent → END (no tool call)
ax.annotate('', xy=(5, 3.35), xytext=(5, 7.05),
    arrowprops=dict(arrowstyle='->', color='#888', lw=1.0,
        connectionstyle="arc3,rad=-0.4"))
ax.text(7.8, 5.2, "no tool\ncall → END", color='#888', fontsize=7.5, ha='center')

ax.text(5, 2.1, "Use: search agents, code executors, RAG",
    color='#888', fontsize=8, ha='center', va='center')

plt.suptitle("LangGraph Graph Topologies", color='white', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("images/langgraph_topologies.png", dpi=150, bbox_inches='tight',
    facecolor=fig.get_facecolor())
plt.show()
print("Saved → images/langgraph_topologies.png")

## 5 — Tool Integration

The standard pattern for a tool-using agent in LangGraph:

| Component | Role |
|---|---|
| `@tool` decorator | Converts a function into a LangChain tool with name + description the LLM sees |
| `llm.bind_tools(tools)` | Attaches tool schemas to the LLM so it can emit `tool_calls` in its response |
| `tools_condition` | Built-in edge function: returns `"tools"` if response has `tool_calls`, else `END` |
| `ToolNode(tools=tools)` | Built-in node: reads `tool_calls` from the last message, executes them, appends results |
| loop: `"tools" → "agent"` | After tool execution, send results back to the LLM for a follow-up response |

The agent loop runs until the LLM returns a response with no `tool_calls`.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, BaseMessage
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

# ── Define tools ──────────────────────────────────────────────────────────────
@tool
def search_web(query: str) -> str:
    """Search the web for current information about any topic."""
    # Production: use DuckDuckGoSearchRun() or TavilySearchResults()
    return f"[Web search results for '{query}']: Python 3.12 released Oct 2023, adds type aliases and f-string improvements."

@tool
def calculate(expression: str) -> str:
    """Safely evaluate a mathematical expression and return the result."""
    try:
        # In production: use a safe math parser, not eval
        allowed = set('0123456789+-*/(). ')
        if not all(c in allowed for c in expression):
            return "Error: only numeric expressions allowed"
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

@tool
def get_date() -> str:
    """Return today's date."""
    from datetime import date
    return str(date.today())

tools = [search_web, calculate, get_date]

# ── Bind tools to LLM ─────────────────────────────────────────────────────────
llm_with_tools = llm.bind_tools(tools)

# ── State ─────────────────────────────────────────────────────────────────────
class ToolAgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# ── Nodes ─────────────────────────────────────────────────────────────────────
def agent_node(state: ToolAgentState) -> dict:
    """LLM decides whether to call tools or answer directly."""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools=tools)  # executes tool_calls from the last AI message

# ── Graph ─────────────────────────────────────────────────────────────────────
agent_builder = StateGraph(ToolAgentState)
agent_builder.add_node("agent", agent_node)
agent_builder.add_node("tools", tool_node)

agent_builder.add_edge(START, "agent")
agent_builder.add_conditional_edges(
    "agent",
    tools_condition,              # → "tools" if tool_calls present, else END
    {"tools": "tools", END: END}
)
agent_builder.add_edge("tools", "agent")   # loop: tool results go back to LLM

agent_graph = agent_builder.compile()

# ── Run ───────────────────────────────────────────────────────────────────────
result = agent_graph.invoke({
    "messages": [HumanMessage(content="What is 1234 * 5678? Also what is today's date?")]
})

print("Conversation trace:")
for msg in result["messages"]:
    mtype = type(msg).__name__
    content = msg.content if msg.content else f"[tool_calls: {[tc['name'] for tc in (msg.tool_calls or [])]}]"
    print(f"  [{mtype}] {str(content)[:120]}")

## 6 — Conditional Routing

`add_conditional_edges(source, router_fn, mapping)`:
- `router_fn(state) -> str` — inspects state and returns a routing key
- `mapping` — dict from routing key to destination node name
- If `mapping` is omitted, the return value is used directly as a node name

**Common routing patterns:**

| Pattern | `router_fn` returns | Typical use |
|---|---|---|
| Binary | `"tools"` or `END` | `tools_condition` — tool call or done |
| Multi-class | `"code"` / `"research"` / `"general"` | Task-type dispatch |
| Loop control | `"continue"` or `"done"` | Max iterations or quality threshold |
| Error handling | `"retry"` or `"fail"` | Validation gates |

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage, SystemMessage
from typing import Annotated, TypedDict, Literal
from langgraph.graph.message import add_messages

# ── State ─────────────────────────────────────────────────────────────────────
class RouterState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    task_type: str   # set by classify node, read by conditional edge

# ── Router function ───────────────────────────────────────────────────────────
def classify_task(state: RouterState) -> Literal["code", "research", "general"]:
    """Inspect last message keywords to decide which expert to route to."""
    text = state["messages"][-1].content.lower()
    if any(kw in text for kw in ["code", "python", "function", "debug", "implement", "syntax"]):
        return "code"
    if any(kw in text for kw in ["research", "find", "search", "what is", "explain", "how does"]):
        return "research"
    return "general"

# ── Expert nodes ─────────────────────────────────────────────────────────────
def code_expert(state: RouterState) -> dict:
    messages = [SystemMessage(content="You are a Python expert. Give precise, working code."),
                *state["messages"]]
    return {"messages": [llm.invoke(messages)], "task_type": "code"}

def research_expert(state: RouterState) -> dict:
    messages = [SystemMessage(content="You are a research assistant. Give factual, cited answers."),
                *state["messages"]]
    return {"messages": [llm.invoke(messages)], "task_type": "research"}

def general_handler(state: RouterState) -> dict:
    return {"messages": [llm.invoke(state["messages"])], "task_type": "general"}

# ── Graph ─────────────────────────────────────────────────────────────────────
router_builder = StateGraph(RouterState)
router_builder.add_node("code_expert",     code_expert)
router_builder.add_node("research_expert", research_expert)
router_builder.add_node("general_handler", general_handler)

# START routes directly via classify_task — no separate classify node needed
router_builder.add_conditional_edges(START, classify_task, {
    "code":     "code_expert",
    "research": "research_expert",
    "general":  "general_handler",
})
router_builder.add_edge("code_expert",     END)
router_builder.add_edge("research_expert", END)
router_builder.add_edge("general_handler", END)

router_graph = router_builder.compile()

# ── Test three queries ────────────────────────────────────────────────────────
test_queries = [
    "Write a Python function to flatten a nested list",
    "Explain how transformers work in NLP",
    "What should I have for dinner?",
]

for q in test_queries:
    result = router_graph.invoke({
        "messages": [HumanMessage(content=q)],
        "task_type": ""
    })
    print(f"Query    : {q}")
    print(f"Routed to: {result['task_type']}")
    print(f"Answer   : {result['messages'][-1].content[:100]}...")
    print()

## 8 — Memory & Checkpointing

Without a checkpointer, every `graph.invoke()` call starts from a blank state. A checkpointer saves state snapshots after each node execution and associates them with a `thread_id`.

| Checkpointer | Storage | Use case |
|---|---|---|
| `MemorySaver` | Python dict (in-process) | Development, testing, single-process apps |
| `SqliteSaver` | SQLite file on disk | Single-server persistence, survives restarts |
| `PostgresSaver` | PostgreSQL | Multi-process, production deployments |
| `RedisSaver` | Redis | High-throughput, distributed deployments |

**Session management:** `config = {"configurable": {"thread_id": "user-123"}}` — all invocations with the same `thread_id` share state history. Different `thread_id` values are isolated sessions.

**What gets saved per checkpoint:**
- Full state snapshot (all fields)
- `checkpoint_id` — unique identifier
- `next` — which node would execute if resumed
- Timestamp and metadata

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, BaseMessage
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

# ── Graph with checkpointer ───────────────────────────────────────────────────
class MemChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def chat_with_memory(state: MemChatState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

mem_builder = StateGraph(MemChatState)
mem_builder.add_node("chat", chat_with_memory)
mem_builder.add_edge(START, "chat")
mem_builder.add_edge("chat", END)

memory = MemorySaver()
persistent_graph = mem_builder.compile(checkpointer=memory)

# ── Session A: multi-turn conversation ────────────────────────────────────────
session_a = {"configurable": {"thread_id": "session-alice"}}

turn1 = persistent_graph.invoke(
    {"messages": [HumanMessage(content="My name is Alice and I work on LLM infrastructure.")]},
    config=session_a
)
print("Turn 1 →", turn1["messages"][-1].content[:80], "...")

turn2 = persistent_graph.invoke(
    {"messages": [HumanMessage(content="What do I work on?")]},
    config=session_a
)
print("Turn 2 →", turn2["messages"][-1].content[:80], "...")

# ── Session B: independent (different thread_id) ──────────────────────────────
session_b = {"configurable": {"thread_id": "session-bob"}}
turn_b = persistent_graph.invoke(
    {"messages": [HumanMessage(content="What do I work on?")]},
    config=session_b
)
print("\nSession B (no prior context):", turn_b["messages"][-1].content[:80], "...")

# ── Inspect saved checkpoints ─────────────────────────────────────────────────
print(f"\nCheckpoints for session-alice:")
for cp in persistent_graph.get_state_history(session_a):
    n_msgs = len(cp.values.get("messages", []))
    cid = cp.config["configurable"].get("checkpoint_id", "")[:12]
    print(f"  checkpoint_id={cid}  messages={n_msgs}  next={cp.next}")

## 9 — Time-Travel Debugging

`get_state_history(config)` returns every checkpoint for a thread, newest first. Each checkpoint has a `.config` that can be passed back to `invoke()` to **replay from that point** — sending a different message from a past state.

**Use cases:**
- Debug why the agent made a wrong decision at step N
- A/B test different responses from the same conversation branch point
- Replay a failed run with a corrected input

In [ ]:
from langchain_core.messages import HumanMessage

# ── Build up a multi-turn session to inspect ──────────────────────────────────
debug_session = {"configurable": {"thread_id": "debug-thread"}}

exchanges = [
    "Hello, I'm building a RAG pipeline.",
    "I'm using ChromaDB as my vector store.",
    "What's the best chunking strategy for PDFs?",
]

for msg in exchanges:
    persistent_graph.invoke({"messages": [HumanMessage(content=msg)]}, config=debug_session)

# ── Inspect full state history ────────────────────────────────────────────────
history = list(persistent_graph.get_state_history(debug_session))
print(f"Total checkpoints: {len(history)}  (newest first)\n")

for i, snap in enumerate(history):
    msgs = snap.values.get("messages", [])
    cid  = snap.config["configurable"].get("checkpoint_id", "N/A")[:16]
    print(f"  [{i}] checkpoint={cid}  messages={len(msgs)}  next={snap.next}")

# ── Time-travel: replay from checkpoint after 1st exchange ───────────────────
# history[-2] is the state after "Hello, I'm building a RAG pipeline." was answered
if len(history) >= 2:
    branch_point = history[-2].config   # config from that earlier checkpoint
    print(f"\nReplaying from checkpoint [{len(history)-2}] with a different follow-up...")

    branched_result = persistent_graph.invoke(
        {"messages": [HumanMessage(content="What are the alternatives to RAG?")]},
        config=branch_point
    )
    print("Branched answer:", branched_result["messages"][-1].content[:120], "...")

## 10 — Human-in-the-Loop

`interrupt(payload)` pauses graph execution at the current node and surfaces `payload` to the caller. Execution resumes when `graph.invoke()` is called again with the same `config` and a `Command(resume=value)`.

**Requires a checkpointer** — the graph must be able to save and restore state across the pause.

```
agent plans action
        │
        ▼
   human_review      ← graph PAUSES here, returns interrupt payload to caller
   interrupt({...})
        │  caller inspects payload, calls graph.invoke(Command(resume={...}))
        ▼
   execute_action     ← resumes with human's decision
        │
        ▼
       END
```

**When to use:** tool execution that has side effects (API calls, file writes, emails), LLM-generated code before running it, any action that is hard to reverse.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

# ── State ─────────────────────────────────────────────────────────────────────
class HITLState(TypedDict):
    messages:       Annotated[list[BaseMessage], add_messages]
    pending_action: str
    approved:       bool

# ── Nodes ─────────────────────────────────────────────────────────────────────
def plan_action(state: HITLState) -> dict:
    """LLM drafts a proposed action based on the user request."""
    response = llm.invoke(state["messages"])
    # Extract a proposed action summary from the response
    action = response.content[:200]
    return {"messages": [response], "pending_action": action}

def human_review(state: HITLState) -> dict:
    """Pause and wait for human approval. Graph halts here until resumed."""
    human_decision = interrupt({
        "message": "Review the proposed action below and approve or reject.",
        "pending_action": state["pending_action"],
    })
    # human_decision is whatever was passed to Command(resume=...)
    approved = human_decision.get("approved", False)
    return {"approved": approved}

def execute_action(state: HITLState) -> dict:
    if state["approved"]:
        msg = AIMessage(content=f"Action approved and executed: {state['pending_action'][:80]}...")
    else:
        msg = AIMessage(content="Action rejected by user. No changes made.")
    return {"messages": [msg]}

# ── Graph ─────────────────────────────────────────────────────────────────────
hitl_builder = StateGraph(HITLState)
hitl_builder.add_node("plan",    plan_action)
hitl_builder.add_node("review",  human_review)
hitl_builder.add_node("execute", execute_action)

hitl_builder.add_edge(START,    "plan")
hitl_builder.add_edge("plan",   "review")
hitl_builder.add_edge("review", "execute")
hitl_builder.add_edge("execute", END)

hitl_graph = hitl_builder.compile(checkpointer=MemorySaver())

# ── Run Phase 1: graph pauses at human_review ─────────────────────────────────
hitl_config = {"configurable": {"thread_id": "hitl-session-1"}}
try:
    result = hitl_graph.invoke(
        {"messages": [HumanMessage(content="Send a summary email to the engineering team about today's deployment.")],
         "pending_action": "", "approved": False},
        config=hitl_config
    )
except Exception as e:
    # GraphInterrupt is raised when interrupt() is called
    print(f"Graph paused — waiting for human review")
    print(f"  Interrupt type : {type(e).__name__}")

# ── Run Phase 2: resume with approval ────────────────────────────────────────
print("\nResuming with human approval → approved=True")
final = hitl_graph.invoke(
    Command(resume={"approved": True}),
    config=hitl_config
)
print("Final message:", final["messages"][-1].content)

# ── Run Phase 3: new session, reject ─────────────────────────────────────────
hitl_config2 = {"configurable": {"thread_id": "hitl-session-2"}}
try:
    hitl_graph.invoke(
        {"messages": [HumanMessage(content="Delete all staging database records.")],
         "pending_action": "", "approved": False},
        config=hitl_config2
    )
except Exception:
    pass  # interrupt raised

print("\nResuming with rejection → approved=False")
final2 = hitl_graph.invoke(
    Command(resume={"approved": False}),
    config=hitl_config2
)
print("Final message:", final2["messages"][-1].content)

## 11 — Summary

**Minimal working graph:**
```python
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage

class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

builder = StateGraph(State)
builder.add_node("chat", lambda s: {"messages": [llm.invoke(s["messages"])]})
builder.add_edge(START, "chat")
builder.add_edge("chat", END)
graph = builder.compile()
graph.invoke({"messages": [HumanMessage(content="Hello")]})
```

**Concept map:**

| Concept | API | Key detail |
|---|---|---|
| State | `TypedDict` / `BaseModel` | Immutable per step; nodes return partial dicts |
| Reducer | `Annotated[T, fn]` | `add_messages` appends; default replaces |
| Node | `add_node("name", fn)` | `fn(state) -> dict` — partial update only |
| Fixed edge | `add_edge(src, dst)` | Always transitions `src → dst` |
| Conditional edge | `add_conditional_edges(src, fn, map)` | `fn(state)` returns key; `map[key]` is next node |
| Tool loop | `ToolNode` + `tools_condition` | Repeats agent→tools until no `tool_calls` |
| Memory | `compile(checkpointer=...)` | Saves snapshot after each node |
| Thread | `{"thread_id": "..."}` in config | Isolates sessions; enables multi-turn memory |
| Time-travel | `get_state_history(config)` | Returns all checkpoints; replay by passing old `.config` |
| Human-in-loop | `interrupt(payload)` + `Command(resume=...)` | Pauses graph; requires checkpointer |

**Common mistakes:**

| Mistake | Fix |
|---|---|
| Node returns full state instead of partial dict | Return only the fields that changed |
| `add_messages` not used → messages overwrite each other | Use `Annotated[list[BaseMessage], add_messages]` |
| `interrupt()` used without checkpointer | Always `compile(checkpointer=MemorySaver())` with HITL |
| Routing function returns node name not in mapping | Ensure all return values appear as keys in the mapping dict |
| Thread state bleeds between users | Use unique `thread_id` per user session |